In [2]:
import os
import pandas as pd
import numpy as np
import re
from pathlib import Path

initialDB_path = Path('../data/initialDB_sequences.csv')
initialDB = pd.read_csv(initialDB_path, sep=',', header=0, comment='#', names=[
    'gene_id', 'transcript_id', '5_UTR', 'CDS', '3_UTR', '5_UTR_len', 'CDS_len', '3_UTR_len'
])

df = initialDB.copy()
df.head()

,gene_id,transcript_id,5_UTR,CDS,3_UTR,5_UTR_len,CDS_len,3_UTR_len
0,ENSG00000186092.7,ENST00000641515.2,CCCAGATCTCTTCAGTTTTTATGCCTCATTCTGTGAAAATTGCTGT...,ATGAAGAAGGTAACTGCAGAGGCTATTTCCTGGAATGAATCAACGA...,NaN,60,978,0
1,ENSG00000284733.2,ENST00000426406.4,NaN,ATGGATGGAGAGAATCACTCAGTGGTATCTGAGTTTTTGTTTCTGG...,TAA,0,936,3
2,ENSG00000284662.2,ENST00000332831.5,NaN,ATGGATGGAGAGAATCACTCAGTGGTATCTGAGTTTTTGTTTCTGG...,TAA,0,936,3
3,ENSG00000187634.14,ENST00000616016.5,GGCGGCGGAGTCTCCCAAGTCCCCGCCGGGCGGGCGCGCGCCAGTG...,ATGCCGGCGGTCAAGAAGGAGTTCCCGGGCCGCGAGGACCTGGCCC...,NaN,509,2532,0
4,ENSG00000188976.12,ENST00000327044.7,GCTTCGGGTTGGTGTC,ATGGCAGCTGCGGGGAGCCGCAAGAGGCGCCTGGCGGAGCTGACGG...,TGAGGCAGCCCATCTGGGGGGCCTGTAGGGGCTGCCGGGCTGGTGG...,16,2247,494


In [3]:
def gc_content(seq):
    if not isinstance(seq, str):
        return np.nan
    seq = seq.strip().upper()
    if not seq:
        return np.nan
    gc = seq.count('G') + seq.count('C')
    return gc / len(seq)

df = initialDB.copy()
df['5_UTR_GC'] = df['5_UTR'].fillna('').map(gc_content)
df['CDS_GC'] = df['CDS'].fillna('').map(gc_content)
df['3_UTR_GC'] = df['3_UTR'].fillna('').map(gc_content)
df['global_GC'] = df[['5_UTR', 'CDS', '3_UTR']].fillna('').agg(''.join, axis=1).map(gc_content)
df.head()

,gene_id,transcript_id,5_UTR,CDS,3_UTR,5_UTR_len,CDS_len,3_UTR_len,5_UTR_GC,CDS_GC,3_UTR_GC,global_GC
0,ENSG00000186092.7,ENST00000641515.2,CCCAGATCTCTTCAGTTTTTATGCCTCATTCTGTGAAAATTGCTGT...,ATGAAGAAGGTAACTGCAGAGGCTATTTCCTGGAATGAATCAACGA...,NaN,60,978,0,0.40000,0.426380,NaN,0.424855
1,ENSG00000284733.2,ENST00000426406.4,NaN,ATGGATGGAGAGAATCACTCAGTGGTATCTGAGTTTTTGTTTCTGG...,TAA,0,936,3,NaN,0.461538,0.000000,0.460064
2,ENSG00000284662.2,ENST00000332831.5,NaN,ATGGATGGAGAGAATCACTCAGTGGTATCTGAGTTTTTGTTTCTGG...,TAA,0,936,3,NaN,0.461538,0.000000,0.460064
3,ENSG00000187634.14,ENST00000616016.5,GGCGGCGGAGTCTCCCAAGTCCCCGCCGGGCGGGCGCGCGCCAGTG...,ATGCCGGCGGTCAAGAAGGAGTTCCCGGGCCGCGAGGACCTGGCCC...,NaN,509,2532,0,0.80943,0.701422,NaN,0.719500
4,ENSG00000188976.12,ENST00000327044.7,GCTTCGGGTTGGTGTC,ATGGCAGCTGCGGGGAGCCGCAAGAGGCGCCTGGCGGAGCTGACGG...,TGAGGCAGCCCATCTGGGGGGCCTGTAGGGGCTGCCGGGCTGGTGG...,16,2247,494,0.62500,0.603471,0.572874,0.598114
